# PySpark Technical Interview Questions - Performance & Optimization

## 📚 Overview

This notebook covers **performance optimization questions** that are critical for senior roles and companies dealing with big data at scale.

**Topics Covered:**
- Broadcast joins vs shuffle joins
- Caching strategies
- Partition optimization
- Predicate pushdown
- Avoiding shuffles
- Data skew handling
- Memory management

**Interview Frequency:** 70%+ of senior/staff engineer interviews, 50%+ for mid-level roles at FAANG.

---

## 💡 Study Tips

1. **Understand WHY, not just HOW** - explain the performance impact
2. **Know the trade-offs** - every optimization has a cost
3. **Use Spark UI** - learn to read query plans (explain())
4. **Benchmark everything** - show before/after metrics

---

## Setup: Create Spark Session with Performance Configs

Notice the additional performance-related configurations!

In [ ]:
from pyspark.sql import SparkSession  # Main entry point
from pyspark.sql.functions import broadcast, col, count, sum as _sum, avg, when, lit  # Transformation and aggregation functions
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType  # Schema types
import time  # For performance benchmarking

# Create Spark Session with Performance Tuning
spark = SparkSession.builder \  # Builder pattern
    .appName("Interview_Performance") \  # Application name
    .master("local[*]") \  # Local mode with all cores
    .config("spark.sql.shuffle.partitions", "4") \  # Shuffle partitions (default=200, we use 4 for local)
    .config("spark.sql.adaptive.enabled", "true") \  # Enable Adaptive Query Execution (Spark 3.0+)
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \  # Auto-combine small partitions
    .config("spark.sql.autoBroadcastJoinThreshold", "10MB") \  # Auto-broadcast tables < 10MB
    .config("spark.default.parallelism", "4") \  # Default parallelism for RDD operations
    .getOrCreate()  # Create or reuse session

spark.sparkContext.setLogLevel("ERROR")  # Suppress INFO/WARN logs

print("✅ Spark Session Created with Performance Configs")  # Confirmation
print(f"Spark Version: {spark.version}")  # Show version
print(f"Shuffle Partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")  # Show shuffle config
print(f"Adaptive Query Execution: {spark.conf.get('spark.sql.adaptive.enabled')}")  # Show AQE status

---

# 🔀 SHUFFLE PARTITIONS & PERFORMANCE

## Key Configuration Explained

```python
.config("spark.sql.shuffle.partitions", "4")
```

This is **ONE OF THE MOST IMPORTANT** performance tuning parameters!

### What It Does:
Controls how many partitions are created during **shuffle operations**:
- JOIN operations
- GROUP BY aggregations
- Window functions with PARTITION BY
- DISTINCT operations
- ORDER BY sorting

### Performance Impact:

| Partitions | Data Size | Result |
|------------|-----------|--------|
| 4 | 1 GB | ✅ Each partition ~250MB - Perfect! |
| 4 | 100 GB | ❌ Each partition ~25GB - OOM! |
| 10,000 | 1 GB | ❌ Each partition ~100KB - Too many tasks! |
| 800 | 100 GB | ✅ Each partition ~128MB - Perfect! |

### Optimization Rules:

1. **Target**: 128-200 MB per partition
2. **Formula**: `Partitions = Data Size MB / 128 MB`
3. **Local dev**: 2-10 partitions
4. **Production (10-100GB)**: 100-200 partitions
5. **Big data (100GB+)**: 500-2000 partitions

### Other Performance Configs:

```python
# Auto-combine small partitions after shuffle
.config("spark.sql.adaptive.coalescePartitions.enabled", "true")

# Automatically broadcast small tables
.config("spark.sql.autoBroadcastJoinThreshold", "10MB")

# Default parallelism for RDDs
.config("spark.default.parallelism", "4")
```

## 💡 Interview Answer:

**Q: "How do you tune Spark performance?"**

**A:** "First, I configure shuffle partitions based on data size - target 128-200MB per partition. For 100GB data, I'd use ~600-800 partitions. I enable AQE for automatic optimization, set broadcast threshold for small dimension tables, and use caching for DataFrames accessed multiple times. I also monitor the Spark UI to identify bottlenecks like data skew or excessive shuffles."

---

---

# QUESTION 1: Broadcast Join vs Regular Join

## 📝 Problem Statement

Given a **large orders table** and a **small customers table**, demonstrate the performance difference between broadcast join and shuffle join.

## 🎯 Interview Focus
- **Frequency**: VERY common at FAANG (90%+)
- **Tests**: Understanding of Spark internals, join strategies
- **Follow-up**: "When should you NOT use broadcast?"

## 💡 Key Concepts

### Shuffle Join (Regular Join)
- Both tables are shuffled across network
- Data redistributed by join key
- Expensive for large tables
- Time: O(N + M) with network overhead

### Broadcast Join
- Small table sent to ALL executors
- No shuffle needed for large table
- Fast for small dimension tables
- Time: O(N) + broadcast overhead

### When to Use Broadcast?
- ✅ Small table < 10MB (configurable)
- ✅ Dimension tables, lookup tables
- ❌ Both tables are large
- ❌ Small table doesn't fit in executor memory

In [ ]:
# Create sample data - simulate large orders and small customers

# LARGE table: 10,000 orders
orders_data = [(i, i % 100, i * 10) for i in range(1, 10001)]
orders_df = spark.createDataFrame(orders_data, ["order_id", "customer_id", "amount"])

# SMALL table: 100 customers
customers_data = [(i, f"Customer_{i}", f"City_{i%10}") for i in range(100)]
customers_df = spark.createDataFrame(customers_data, ["customer_id", "name", "city"])

print(f"Orders: {orders_df.count()} rows")
print(f"Customers: {customers_df.count()} rows")

print("\nSample Orders:")
orders_df.show(5)

print("Sample Customers:")
customers_df.show(5)

In [ ]:
# METHOD 1: Regular Shuffle Join (Default)

print("\n🐌 Regular Shuffle Join:")
print("=" * 50)

start_time = time.time()

# Regular join - Spark decides strategy automatically
shuffle_join = orders_df.join(customers_df, "customer_id")
result_count = shuffle_join.count()  # Action to trigger execution

shuffle_time = time.time() - start_time

print(f"Result: {result_count} rows")
print(f"Time: {shuffle_time:.4f} seconds")

# Show query plan
print("\n📊 Physical Plan (look for 'Exchange' = shuffle):")
shuffle_join.explain(mode="simple")

In [ ]:
# METHOD 2: Broadcast Join (Optimized)

print("\n🚀 Broadcast Join:")
print("=" * 50)

start_time = time.time()

# Explicitly broadcast the small customers table
# This forces Spark to use broadcast join strategy
broadcast_join = orders_df.join(broadcast(customers_df), "customer_id")
result_count = broadcast_join.count()

broadcast_time = time.time() - start_time

print(f"Result: {result_count} rows")
print(f"Time: {broadcast_time:.4f} seconds")

# Show query plan
print("\n📊 Physical Plan (look for 'BroadcastHashJoin'):")
broadcast_join.explain(mode="simple")

In [ ]:
# PERFORMANCE COMPARISON

print("\n" + "=" * 60)
print("📊 PERFORMANCE COMPARISON")
print("=" * 60)

print(f"\nShuffle Join Time:    {shuffle_time:.4f} seconds")
print(f"Broadcast Join Time:  {broadcast_time:.4f} seconds")

if shuffle_time > 0:
    speedup = (shuffle_time / broadcast_time)
    improvement = ((shuffle_time - broadcast_time) / shuffle_time) * 100
    print(f"\n🚀 Speedup: {speedup:.2f}x faster")
    print(f"📈 Improvement: {improvement:.1f}% reduction in time")

print("\n💡 Why is broadcast faster?")
print("   - No shuffle of large orders table")
print("   - Customers table sent to all executors once")
print("   - Join happens locally on each executor")
print("   - Reduces network I/O significantly")

### ✅ Key Takeaways - Question 1

1. **Use `broadcast()` for small tables** (< 10MB default, < 100MB max)
2. **Check query plan** with `.explain()` to verify broadcast is used
3. **Configure threshold**: `spark.sql.autoBroadcastJoinThreshold`
4. **Interview Answer**: "Broadcast join avoids shuffle by sending small table to all executors"

**Common Follow-ups:**
- Q: "What if small table is 1GB?" → A: Don't broadcast, will cause OOM
- Q: "Can you broadcast multiple tables?" → A: Yes, but watch memory
- Q: "How to disable auto-broadcast?" → A: Set threshold to -1

---

# QUESTION 2: Caching Strategies

## 📝 Problem Statement

Demonstrate when and how to use **caching** to improve performance for iterative operations.

## 🎯 Interview Focus
- **Frequency**: Common (60%+ of interviews)
- **Tests**: Understanding of lazy evaluation, DAG optimization
- **Follow-up**: "cache() vs persist() - what's the difference?"

## 💡 Key Concepts

### When to Cache?
- ✅ DataFrame used multiple times
- ✅ Expensive transformations (joins, aggregations)
- ✅ Iterative algorithms (ML training)
- ❌ Large DataFrames that don't fit in memory
- ❌ DataFrame used only once

### Storage Levels
- `MEMORY_ONLY`: Fastest, but can spill if OOM
- `MEMORY_AND_DISK`: Safer, spills to disk
- `MEMORY_ONLY_SER`: Serialized, saves memory
- `DISK_ONLY`: Slow, but persistent

In [ ]:
# Create a DataFrame with expensive transformation
from pyspark.sql.functions import rand, round as _round

# Generate 1 million rows with random data
large_df = spark.range(0, 1000000) \
    .withColumn("value", (_round(rand() * 1000, 2))) \
    .withColumn("category", (col("id") % 100).cast("string"))

print("Created large DataFrame (1M rows)")
print(f"Partitions: {large_df.rdd.getNumPartitions()}")

In [ ]:
# SCENARIO: Perform multiple aggregations WITHOUT caching

print("\n🐌 WITHOUT CACHING:")
print("=" * 50)

start_time = time.time()

# Aggregation 1: Total count
total = large_df.count()
print(f"Total rows: {total}")

# Aggregation 2: Sum by category
sum_result = large_df.groupBy("category").agg(_sum("value").alias("total")).count()
print(f"Categories: {sum_result}")

# Aggregation 3: Average by category
avg_result = large_df.groupBy("category").agg(avg("value").alias("avg")).count()
print(f"Average groups: {avg_result}")

time_without_cache = time.time() - start_time
print(f"\nTime WITHOUT cache: {time_without_cache:.4f} seconds")

print("\n⚠️ Notice: DataFrame is recomputed for EACH action!")

In [ ]:
# SCENARIO: Perform multiple aggregations WITH caching

print("\n🚀 WITH CACHING:")
print("=" * 50)

# Cache the DataFrame in memory
# First action will materialize and cache it
cached_df = large_df.cache()

start_time = time.time()

# Aggregation 1: Total count (materializes cache)
total = cached_df.count()
print(f"Total rows: {total}")

# Aggregation 2: Sum by category (reads from cache!)
sum_result = cached_df.groupBy("category").agg(_sum("value").alias("total")).count()
print(f"Categories: {sum_result}")

# Aggregation 3: Average by category (reads from cache!)
avg_result = cached_df.groupBy("category").agg(avg("value").alias("avg")).count()
print(f"Average groups: {avg_result}")

time_with_cache = time.time() - start_time
print(f"\nTime WITH cache: {time_with_cache:.4f} seconds")

print("\n✅ DataFrame computed once, reused multiple times!")

In [ ]:
# PERFORMANCE COMPARISON & CLEANUP

print("\n" + "=" * 60)
print("📊 CACHING PERFORMANCE COMPARISON")
print("=" * 60)

print(f"\nWithout Cache:  {time_without_cache:.4f} seconds")
print(f"With Cache:     {time_with_cache:.4f} seconds")

if time_without_cache > time_with_cache:
    speedup = time_without_cache / time_with_cache
    improvement = ((time_without_cache - time_with_cache) / time_without_cache) * 100
    print(f"\n🚀 Speedup: {speedup:.2f}x faster")
    print(f"📈 Improvement: {improvement:.1f}% reduction")

# IMPORTANT: Always unpersist when done!
cached_df.unpersist()
print("\n🧹 Cache cleared (unpersisted)")

print("\n💡 Why cache helped:")
print("   - DataFrame reused 3 times")
print("   - Computed once, stored in memory")
print("   - Subsequent operations read from memory")
print("   - Avoided recomputing from source")

In [ ]:
# BONUS: cache() vs persist() Comparison

from pyspark import StorageLevel

print("\n📚 CACHE vs PERSIST:")
print("=" * 50)

print("\ncache() → Equivalent to persist(StorageLevel.MEMORY_ONLY)")
print("persist() → Allows you to choose storage level\n")

print("Available Storage Levels:")
print("  - MEMORY_ONLY:          Fast, but may evict if OOM")
print("  - MEMORY_AND_DISK:      Safer, spills to disk")
print("  - MEMORY_ONLY_SER:      Serialized, saves space")
print("  - DISK_ONLY:            Slow but persistent")
print("  - MEMORY_AND_DISK_SER:  Balanced approach")

# Example of using persist with custom storage level
print("\nExample: persist with MEMORY_AND_DISK")
example_df = spark.range(100)
example_df.persist(StorageLevel.MEMORY_AND_DISK)
print("DataFrame persisted with MEMORY_AND_DISK strategy")
example_df.unpersist()

### ✅ Key Takeaways - Question 2

1. **Cache when**: DataFrame used 2+ times, expensive to recompute
2. **Don't cache**: Single-use DataFrames, data larger than memory
3. **Always unpersist**: Free up memory when done
4. **Use persist()** for custom storage levels

**Interview Answers:**
- Q: "cache() vs persist()?" → A: "cache() is persist(MEMORY_ONLY)"
- Q: "When to use MEMORY_AND_DISK?" → A: "When data might not fit in memory"
- Q: "How to check cache usage?" → A: "Use Spark UI Storage tab"

---

# Continue practicing...

The remaining optimization questions follow the same pattern:
1. Problem statement
2. Before/after code with benchmarks
3. Query plan analysis
4. Key takeaways

**Practice Questions 3-10:**
- Q3: Partition Optimization
- Q4: Filter Pushdown
- Q5: Avoiding Shuffles
- Q6: Column Pruning
- Q7: Predicate Pushdown with Parquet
- Q8: Avoiding UDFs
- Q9: Data Skew Handling (Salting)
- Q10: Memory Management

💡 **Interview Tip**: Always explain the *why* behind optimizations, not just the *how*.